In [0]:
# Step 1: Read data from the table
df = spark.table("main.default.autoloader_csv_demo")

# Step 2: Write as Parquet to Volumes
parquet_path = "/Volumes/main/default/raw_data_volume/parquet_files/user_data/"
df.write.mode("overwrite").parquet(parquet_path)

print(f"Parquet files written to: {parquet_path}")

In [0]:
# Read Parquet files and create table
parquet_path = "/Volumes/main/default/raw_data_volume/parquet_files/user_data/"
df_parquet = spark.read.parquet(parquet_path)

# Save as Delta table
df_parquet.write.mode("overwrite").saveAsTable("main.default.user_data_external")

print(f"Table created: main.default.user_data_external")
print(f"Total rows: {df_parquet.count()}")

In [0]:
from delta.tables import DeltaTable

table_name = "main.default.user_data_external"
delta_table = DeltaTable.forName(spark, table_name)
spark.sql(f"OPTIMIZE {table_name} ZORDER BY (city, ingestion_time)")

In [0]:
# Without Z-ORDER: Scans ALL files
# With Z-ORDER: Scans ONLY relevant files (10-100x faster)

table_name = "main.default.user_data_external"
history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
detail_df = spark.sql(f"DESCRIBE DETAIL {table_name}")

In [0]:
%sql
-- Verify the external table
SELECT * FROM main.default.user_data_external LIMIT 10;

In [0]:
%sql
-- Step 1: Compact small files into larger ones for better read performance
OPTIMIZE main.default.user_data_external;

-- Step 2: Z-ORDER optimization
-- Z-ORDER co-locates related data in the same files using multi-dimensional clustering
-- Benefits:
--   - Faster queries with WHERE filters on Z-ORDERed columns
--   - Data skipping: reads only relevant files, not entire table
--   - Best for columns frequently used in WHERE clauses

-- Example: If you often query by city and ingestion_time:
--   SELECT * FROM table WHERE city = 'Bangalore' AND ingestion_time > '2026-01-01'
-- Z-ORDER will cluster rows with same city + time range together in files

OPTIMIZE main.default.user_data_external ZORDER BY (city, ingestion_time);

-- Step 3: Collect column statistics for query optimizer
ANALYZE TABLE main.default.user_data_external COMPUTE STATISTICS FOR ALL COLUMNS;

-- When to use Z-ORDER:
-- ✓ High-cardinality columns (many unique values): city, user_id, date
-- ✓ Columns in WHERE clauses of frequent queries
-- ✓ Columns used in JOIN conditions
-- ✗ Low-cardinality columns (few values): status with 2-3 values
-- ✗ Columns you rarely filter on

In [0]:
%sql
-- View table optimization history
-- Shows OPTIMIZE and Z-ORDER operations performed
DESCRIBE HISTORY main.default.user_data_external;

-- View detailed table information including file count
-- After OPTIMIZE: fewer, larger files
-- Before OPTIMIZE: many small files
DESCRIBE DETAIL main.default.user_data_external;

In [0]:
%sql
-- Query filtered by Z-ORDERed columns (city, ingestion_time)
-- After Z-ORDER, this query will:
--   1. Skip files that don't contain 'Bangalore'
--   2. Skip files outside the time range
--   3. Read only relevant files (data skipping)

SELECT 
    city,
    COUNT(*) as user_count,
    MIN(ingestion_time) as first_ingestion,
    MAX(ingestion_time) as last_ingestion
FROM main.default.user_data_external
WHERE city = 'Bangalore'
  AND ingestion_time >= '2026-03-01'
GROUP BY city;

-- Check optimization stats:
-- DESCRIBE HISTORY main.default.user_data_external;

In [0]:
# View Delta Table Optimization History Using Python
# Shows what OPTIMIZE and Z-ORDER operations were performed

from delta.tables import DeltaTable
import pandas as pd

table_name = "main.default.user_data_external"

print("=== Delta Table History ===")
history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")

# Show recent operations
recent_ops = history_df.select(
    "version", 
    "timestamp", 
    "operation", 
    "operationParameters"
).limit(10)

print("\nRecent Operations:")
for row in recent_ops.collect():
    print(f"\nVersion: {row['version']}")
    print(f"Time: {row['timestamp']}")
    print(f"Operation: {row['operation']}")
    if row['operation'] == 'OPTIMIZE':
        params = row['operationParameters']
        if 'zOrderBy' in params:
            print(f"Z-ORDER columns: {params['zOrderBy']}")
        print("  → Files compacted and data clustered")

print("\n=== Table Details ===")
detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]

print(f"\nTable: {detail['name']}")
print(f"Format: {detail['format']}")
print(f"Location: {detail['location']}")
print(f"Number of files: {detail['numFiles']}")
print(f"Total size: {detail['sizeInBytes']:,} bytes ({detail['sizeInBytes'] / 1024:.2f} KB)")

print("\n=== Column Statistics ===")
stats_df = spark.sql(f"DESCRIBE EXTENDED {table_name}")

# Show table statistics
print("\nTable-level statistics collected for query optimization")
print("These help Spark choose the best query execution plan")

print("\n=== Z-ORDER Summary ===")
print("✓ OPTIMIZE compacts small files → fewer files, faster reads")
print("✓ Z-ORDER clusters data by columns → data skipping enabled")
print("✓ ANALYZE collects statistics → better query plans")
print("\nResult: Faster queries, lower costs, better performance!")

In [0]:
# Demonstrate Z-ORDER Performance Benefits
# After Z-ORDER, queries filtered by (city, ingestion_time) will be MUCH faster

import time
from pyspark.sql.functions import col

table_name = "main.default.user_data_external"

print("=== How Z-ORDER Improves Performance ===")
print("\nWithout Z-ORDER:")
print("  File 1: [Bangalore, Mumbai, Chennai, Pune] - random mix")
print("  File 2: [Bangalore, Mumbai, Chennai, Pune] - random mix")
print("  File 3: [Bangalore, Mumbai, Chennai, Pune] - random mix")
print("  Query: WHERE city = 'Bangalore'")
print("  Result: Must scan ALL 3 files")

print("\nWith Z-ORDER BY (city):")
print("  File 1: [Bangalore, Bangalore, Bangalore] - clustered")
print("  File 2: [Mumbai, Chennai] - clustered")
print("  File 3: [Pune] - clustered")
print("  Query: WHERE city = 'Bangalore'")
print("  Result: Only scans File 1 (data skipping!)\n")

# Query that benefits from Z-ORDER
print("=== Running optimized query ===")
start_time = time.time()

df_result = spark.sql(f"""
    SELECT 
        city,
        COUNT(*) as user_count,
        MIN(ingestion_time) as first_seen,
        MAX(ingestion_time) as last_seen
    FROM {table_name}
    WHERE city = 'Bangalore'
      AND ingestion_time >= '2026-03-01'
    GROUP BY city
""")

result = df_result.collect()
query_time = time.time() - start_time

print(f"Query completed in: {query_time:.4f} seconds")
print(f"Results: {len(result)} rows")
if result:
    print(f"\nData for Bangalore:")
    for row in result:
        print(f"  City: {row['city']}")
        print(f"  User count: {row['user_count']}")
        print(f"  First seen: {row['first_seen']}")
        print(f"  Last seen: {row['last_seen']}")

print("\n=== Key Benefit: Data Skipping ===")
print("Z-ORDER enables 'data skipping' - reading only relevant files")
print("Performance improvement: 10-100x faster for large tables")
print("Cost savings: Read fewer files = lower compute costs")

In [0]:
# Z-ORDER Optimization Using Python/PySpark
# Z-ORDER is a multi-dimensional clustering technique that organizes data
# to improve query performance by co-locating related data in the same files

from delta.tables import DeltaTable

# Reference the Delta table
table_name = "main.default.user_data_external"
delta_table = DeltaTable.forName(spark, table_name)

print("=== Step 1: Check table details BEFORE optimization ===")
detail_before = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
print(f"Files before: {detail_before['numFiles']}")
print(f"Size before: {detail_before['sizeInBytes']} bytes\n")

# Step 2: Run OPTIMIZE to compact small files
print("=== Step 2: Running OPTIMIZE (compact files) ===")
spark.sql(f"OPTIMIZE {table_name}")
print("✓ Files compacted\n")

# Step 3: Run Z-ORDER by columns frequently used in WHERE clauses
print("=== Step 3: Running Z-ORDER BY (city, ingestion_time) ===")
print("Z-ORDER clusters data by specified columns for faster filtering")
print("Example: WHERE city = 'Bangalore' AND ingestion_time > '2026-01-01'\n")

spark.sql(f"OPTIMIZE {table_name} ZORDER BY (city, ingestion_time)")
print("✓ Data Z-ORDERed\n")

# Step 4: Collect statistics for query optimizer
print("=== Step 4: Collecting column statistics ===")
spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS FOR ALL COLUMNS")
print("✓ Statistics collected\n")

# Step 5: Check results AFTER optimization
print("=== Step 5: Table details AFTER optimization ===")
detail_after = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
print(f"Files after: {detail_after['numFiles']}")
print(f"Size after: {detail_after['sizeInBytes']} bytes")
print(f"\nReduction: {detail_before['numFiles'] - detail_after['numFiles']} fewer files")

print("\n=== Z-ORDER Best Practices ===")
print("✓ Use on high-cardinality columns (city, user_id, date)")
print("✓ Choose columns in frequent WHERE clauses")
print("✓ Order matters: put most selective column first")
print("✗ Avoid low-cardinality columns (status: active/inactive)")
print("✗ Don't Z-ORDER on rarely queried columns")